# RAG: Baseline Bi-Encoder Retrieval vs. Cross-Encoder Re-Ranking

**Corpus:** *Twilight Imperium: Fourth Edition, Living Rules Reference v1.3* (Fantasy Flight Games, 38 pages), a dense rulebook with
89 numbered glossary rules, many of them “cannot … except …” clauses, plus an errata/FAQ section.

**What this notebook does**

| Step | What |
|---|---|
| 1 | Load the PDF and chunk it in a rule-aware way (one shared corpus for both pipelines) |
| 2 | Index the chunks in ChromaDB with a bi-encoder (`all-MiniLM-L6-v2`) |
| 3 | **Pipeline 1:** cosine top-10, then top-3 go to the LLM, giving **Answer 1** |
| 4 | **Pipeline 2:** the same top-10 are re-scored by a cross-encoder (`ms-marco-MiniLM-L-6-v2`), and the re-ranked top-3 go to the LLM, giving **Answer 2** |
| 5 | Stress tests: keyword traps, exceptions, negations, and chunks buried at ranks 4–10 |
| 6 | Latency profiling and a scaling experiment (cross-encoder over the whole corpus) |

LLM: Google Gemini (`google-genai` SDK). The API key is read from a local `.env` file (`GEMINI_API_KEY=...`) and is never committed.

## 0. Setup: imports and configuration

In [1]:
import os, re, json, time, statistics, warnings
from pathlib import Path
warnings.filterwarnings("ignore", message="IProgress not found")   # cosmetic tqdm warning in notebooks

import numpy as np
import pandas as pd
import torch
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder
import chromadb
from chromadb.config import Settings
from dotenv import load_dotenv
from google import genai
from google.genai import types

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

In [2]:
# ---- configuration (everything tunable lives here) ----
PDF_PATH      = Path("data/ti4_living_rules_reference_v1_3.pdf")
RESULTS_DIR   = Path("results"); RESULTS_DIR.mkdir(exist_ok=True)

BI_ENCODER    = "sentence-transformers/all-MiniLM-L6-v2"   # dense bi-encoder for first-stage retrieval
CROSS_ENCODER = "cross-encoder/ms-marco-MiniLM-L-6-v2"     # re-ranker for the second stage
LLM_MODEL     = "gemini-3.5-flash"                         # generator (same model + prompt for both pipelines)

TOP_K = 10    # candidates retrieved by the bi-encoder
TOP_N = 3     # chunks passed to the LLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print("torch", torch.__version__, "| device:", DEVICE)

torch 2.11.0+cpu | device: cpu


## 1. Load and chunk the PDF

The rulebook layout drives the chunking:

* **pp. 6–31, Glossary:** 89 numbered topics (`19 COMMAND TOKENS`), each with numbered rules (`19.1`, `19.2`, …).
  Because the pages use two columns, pypdf sometimes emits a later heading before an earlier one. So every rule line is
  assigned to a section **from its own number** (`19.6` belongs to section 19), not from the last heading seen.
* **pp. 4–5, Setup steps,** and **pp. 34–38, Errata / FAQ:** a new unit starts at each `STEP n` or `Q:` line.
* **Skipped:** cover, credits, the changelog (p. 1–3) and the alphabetical index (pp. 32–33). The index is just keyword lists
  pointing at rule numbers, so it would add pure noise.
* A **Quick Reference** sidebar on p. 11 is extracted twice by pypdf, in the middle of rule 19. It is cut out and kept as one chunk.

Consecutive units from the same section are packed together up to ~180 words, and a single unit longer than that is split with a 30-word
overlap. MiniLM truncates input at 256 word-pieces, so chunks must stay below that. Each chunk is prefixed with its section title
(e.g. `[Rule 19 Command Tokens]`) so it is self-describing for both the retriever and the LLM.

In [3]:
import re
from pypdf import PdfReader

SETUP_PAGES = range(4, 6)        # 1-indexed: setup steps
GLOSSARY_PAGES = range(6, 32)    # numbered rules 1-89
FAQ_PAGES = range(34, 39)        # errata + FAQ
MAX_WORDS = 180
OVERLAP = 30

HEADING_RE = re.compile(r"^(\d{1,2}) ([A-Z][A-Z ,'’()\-—]{2,})$")
RULE_RE = re.compile(r"^(\d{1,2})\.(\d{1,2})\s")
STEP_RE = re.compile(r"^STEP \d+")
QA_RE = re.compile(r"^Q:")
TOPIC_RE = re.compile(r"^[A-Z][A-Z “”\"'’,\-—]{3,}$")


def clean_lines(text):
    """Drop page numbers / blank lines, fix soft hyphenation."""
    text = re.sub(r"-\n(?=[a-z])", "", text)
    out = []
    for line in text.splitlines():
        line = re.sub(r"\s+", " ", line).strip()
        if not line or line.isdigit():
            continue
        out.append(line)
    return out


def load_pages(path):
    reader = PdfReader(path)
    return {i + 1: clean_lines(p.extract_text() or "") for i, p in enumerate(reader.pages)}


def pop_quick_reference(lines):
    """Page 11 has a 'Quick Reference' sidebar that pypdf extracts twice, in the
    middle of rule 19. Cut both copies out and return (remaining lines, one copy)."""
    qr = [i for i, l in enumerate(lines) if l == "QUICK REFERENCE"]
    if not qr:
        return lines, []
    start = max(i for i in range(qr[0]) if lines[i] == "FLEET")
    return lines[:start] + lines[qr[-1] + 1:], lines[start:qr[0]]


def glossary_units(pages):
    """Split the glossary into rule-level units, each tagged with its section number."""
    titles, units, cur = {}, [], None
    sec = 0
    for pg in GLOSSARY_PAGES:
        for line in pages[pg]:
            if pg == 6 and not titles and not HEADING_RE.match(line):
                continue  # skip the glossary preamble before "1 ABILITIES"
            h, r = HEADING_RE.match(line), RULE_RE.match(line)
            if h and int(h.group(1)) <= 89:
                sec = int(h.group(1))
                titles[sec] = h.group(2).title()
                cur = {"sec": sec, "page": pg, "lines": []}
                units.append(cur)
                continue
            if r:
                sec = int(r.group(1))
                cur = {"sec": sec, "page": pg, "lines": [line]}
                units.append(cur)
                continue
            if cur is not None:
                cur["lines"].append(line)
    return titles, units


def split_long(units, max_words=MAX_WORDS, overlap=OVERLAP):
    """Break any single unit longer than max_words into overlapping word windows."""
    out = []
    for u in units:
        words = " ".join(u["lines"]).split()
        if len(words) <= max_words:
            out.append(u)
            continue
        step = max_words - overlap
        for s in range(0, len(words) - overlap, step):
            out.append({**u, "lines": [" ".join(words[s:s + max_words])]})
    return out


def pack(units, label_fn, max_words=MAX_WORDS):
    """Greedily merge consecutive units that share a label until max_words."""
    units = split_long(units, max_words)
    chunks, buf, buf_label, buf_page = [], [], None, None

    def flush():
        if buf:
            chunks.append({"label": buf_label, "page": buf_page, "text": " ".join(buf)})

    for u in units:
        label, text = label_fn(u), " ".join(u["lines"])
        n = len(text.split())
        if buf and (label != buf_label or len(" ".join(buf).split()) + n > max_words):
            flush()
            buf = []
        if not buf:
            buf_label, buf_page = label, u["page"]
        buf.append(text)
    flush()
    return chunks


def simple_units(pages, page_range, start_re, topic_re=None):
    """Units for setup / FAQ pages: a new unit begins at each start_re line."""
    units, cur, topic = [], None, ""
    for pg in page_range:
        for line in pages[pg]:
            if topic_re and topic_re.match(line) and not start_re.match(line):
                topic = line.title()
                cur = None
                continue
            if start_re.match(line) or cur is None:
                cur = {"topic": topic, "page": pg, "lines": []}
                units.append(cur)
            cur["lines"].append(line)
    return units


def build_chunks(path):
    pages = load_pages(path)
    pages[11], quick_ref = pop_quick_reference(pages[11])
    titles, gunits = glossary_units(pages)
    chunks = [{"label": None, "section": "Quick Reference", "page": 11,
               "text": " ".join(quick_ref)}] if quick_ref else []
    for c in pack(gunits, lambda u: u["sec"]):
        c["section"] = f"Rule {c['label']} {titles.get(c['label'], '')}".strip()
        chunks.append(c)
    for c in pack(simple_units(pages, SETUP_PAGES, STEP_RE), lambda u: "setup"):
        c["section"] = "Game Setup"
        chunks.append(c)
    for c in pack(simple_units(pages, FAQ_PAGES, QA_RE, TOPIC_RE), lambda u: u["topic"]):
        c["section"] = f"Errata/FAQ {c['label']}".strip()
        chunks.append(c)
    for i, c in enumerate(chunks):
        c["id"] = i
        c.pop("label")
        # the section title is prepended so each chunk is self-describing
        c["text"] = f"[{c['section']}] {c['text']}"
    return chunks

In [4]:
chunks = build_chunks(PDF_PATH)
texts = [c["text"] for c in chunks]

lens = [len(t.split()) for t in texts]
print(f"{len(chunks)} chunks | words per chunk: min {min(lens)}, median {statistics.median(lens)}, max {max(lens)}")
json.dump(chunks, open(RESULTS_DIR / "chunks.json", "w"), indent=1, ensure_ascii=False)

pd.DataFrame(chunks)[["id", "page", "section", "text"]].sample(5, random_state=1)

225 chunks | words per chunk: min 9, median 120, max 186


,id,page,section,text
39,39,11,Rule 18 Command Sheet,"[Rule 18 Command Sheet] Each player has a command sheet that contains a strategy pool, a tactic pool, a fleet pool, ..."
166,166,30,Rule 84 Transport,[Rule 84 Transport] 84.3 Fighters and ground forces cannot be picked up from a system that contains one of their fac...
93,93,19,Rule 51 Neighbors,[Rule 51 Neighbors] Two players are neighbors if they both have a unit or control a planet in the same system. They ...
62,62,14,Rule 30 Diplomacy (Strategy Card),[Rule 30 Diplomacy (Strategy Card)] 30.3 After the active player resolves the primary ability of the “Diplomacy” str...
195,195,34,Errata/FAQ Hyper Metabolism,[Errata/FAQ Hyper Metabolism] The text of the “Hyper Metabolism” technology should read as follows: “During the stat...


## 2. Bi-encoder index (ChromaDB)

The bi-encoder embeds every chunk **independently** into a 384-d vector, which is done once, offline. Vectors are L2-normalised and stored
in an in-memory ChromaDB collection that uses cosine space (Chroma returns `distance = 1 − cosine`).

In [5]:
bi_encoder = SentenceTransformer(BI_ENCODER, device=DEVICE)

t0 = time.perf_counter()
chunk_emb = bi_encoder.encode(texts, normalize_embeddings=True, batch_size=32, show_progress_bar=False)
index_encode_s = time.perf_counter() - t0

client = chromadb.EphemeralClient(settings=Settings(anonymized_telemetry=False))
try:
    client.delete_collection("ti4_rules")          # makes the cell safe to re-run
except Exception:
    pass
collection = client.create_collection("ti4_rules", configuration={"hnsw": {"space": "cosine"}},
                                      embedding_function=None)   # we supply our own vectors
collection.add(ids=[str(c["id"]) for c in chunks], embeddings=chunk_emb.tolist(), documents=texts,
               metadatas=[{"page": c["page"], "section": c["section"]} for c in chunks])

print(f"embedded {len(texts)} chunks in {index_encode_s:.2f}s | dim = {chunk_emb.shape[1]} | collection size = {collection.count()}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12738.12it/s]

embedded 225 chunks in 4.53s | dim = 384 | collection size = 225


## 3. Pipeline 1: baseline retrieval (bi-encoder top-10, then top-3)

In [6]:
def baseline_retrieve(query, k=TOP_K):
    # Embed the query with the same bi-encoder and return the k most similar chunks (by cosine).
    q_emb = bi_encoder.encode(query, normalize_embeddings=True)
    res = collection.query(query_embeddings=[q_emb.tolist()], n_results=k, include=["distances"])
    return [{"chunk_id": int(cid), "bi_rank": r + 1, "bi_score": 1 - dist}
            for r, (cid, dist) in enumerate(zip(res["ids"][0], res["distances"][0]))]

# sanity check: Chroma's HNSW result must agree with an exact numpy cosine search on this small corpus
_q = "Can ships move through a nebula?"
_exact = np.argsort(-(chunk_emb @ bi_encoder.encode(_q, normalize_embeddings=True)))[:TOP_K].tolist()
assert [c["chunk_id"] for c in baseline_retrieve(_q)] == _exact
pd.DataFrame(baseline_retrieve(_q)).assign(section=lambda d: [chunks[i]["section"] for i in d.chunk_id])

,chunk_id,bi_rank,bi_score,section
0,84,1,0.650027,Rule 50 Nebula
1,151,2,0.465016,Rule 75 Supernova
2,82,3,0.401888,Rule 49 Movement
3,0,4,0.374649,Quick Reference
4,172,5,0.355615,Rule 84 Transport
5,89,6,0.337536,Rule 49 Movement
6,148,7,0.315227,Rule 78 Tactical Action
7,29,8,0.300840,Rule 11 Asteroid Field
8,201,9,0.299086,Errata/FAQ General
9,222,10,0.298607,Errata/FAQ The Ghosts Of Creuss


## 4. Pipeline 2: cross-encoder re-ranking of the same top-10

The cross-encoder reads each **(query, chunk) pair together** in one transformer pass, so every query token can attend to every chunk
token. It outputs one relevance logit per pair. It only ever sees the 10 candidates from Pipeline 1, so both pipelines draw on exactly the
same candidate pool.

In [7]:
cross_encoder = CrossEncoder(CROSS_ENCODER, device=DEVICE)

def rerank(query, candidates):
    # Score all (query, chunk) pairs jointly, then sort candidates by cross-encoder score (descending).
    pairs = [(query, chunks[c["chunk_id"]]["text"]) for c in candidates]
    scores = cross_encoder.predict(pairs, show_progress_bar=False)
    out = sorted(({**c, "ce_score": float(s)} for c, s in zip(candidates, scores)),
                 key=lambda c: c["ce_score"], reverse=True)
    for r, c in enumerate(out):
        c["ce_rank"] = r + 1
    return out

cands = baseline_retrieve(_q)
pd.DataFrame(rerank(_q, cands)).assign(section=lambda d: [chunks[i]["section"] for i in d.chunk_id])

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 8697.41it/s]

,chunk_id,bi_rank,bi_score,ce_score,ce_rank,section
0,84,1,0.650027,9.022667,1,Rule 50 Nebula
1,0,4,0.374649,4.973814,2,Quick Reference
2,151,2,0.465016,0.970473,3,Rule 75 Supernova
3,82,3,0.401888,0.063535,4,Rule 49 Movement
4,29,8,0.300840,-0.246457,5,Rule 11 Asteroid Field
5,172,5,0.355615,-2.837964,6,Rule 84 Transport
6,222,10,0.298607,-2.843670,7,Errata/FAQ The Ghosts Of Creuss
7,201,9,0.299086,-4.170620,8,Errata/FAQ General
8,89,6,0.337536,-4.326828,9,Rule 49 Movement
9,148,7,0.315227,-6.673171,10,Rule 78 Tactical Action


## 5. Generation: one identical prompt for both pipelines

The prompt tells the model to answer **only from the excerpts** and to say so when they do not contain the answer. Gemini has probably
seen Twilight Imperium rules during pre-training. Without this instruction it could “fix” a bad retrieval from memory and hide the
difference between the pipelines. Temperature is 0 and thinking is disabled, so the only thing that differs between Answer 1 and Answer 2
is the context.

In [8]:
load_dotenv()
assert os.getenv("GEMINI_API_KEY"), "Put GEMINI_API_KEY=... in a .env file next to this notebook"
llm = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

PROMPT_TEMPLATE = '''You are a rules assistant for the board game Twilight Imperium: Fourth Edition.
Answer the question using ONLY the rulebook excerpts below. Do not use any outside knowledge of the game.
If the excerpts do not contain the information needed, reply exactly: "The provided excerpts do not cover this."
Answer in 2-4 sentences and cite the chunk ids you relied on, e.g. [Chunk 12].

Rulebook excerpts:
{context}

Question: {question}
Answer:'''

def build_prompt(question, ctx):
    context = "\n\n".join(f"[Chunk {c['chunk_id']}, p.{chunks[c['chunk_id']]['page']}] {chunks[c['chunk_id']]['text']}" for c in ctx)
    return PROMPT_TEMPLATE.format(context=context, question=question)

# Responses are cached on disk (keyed by model + prompt), so re-running the notebook reproduces the same answers
# without spending API quota. The free tier allows only ~20 requests per model per day.
CACHE_PATH = RESULTS_DIR / "llm_cache.json"
llm_cache = json.load(open(CACHE_PATH)) if CACHE_PATH.exists() else {}

def generate(question, ctx, retries=6):
    # Call Gemini with the shared prompt; back off and retry on per-minute rate limits / transient errors.
    prompt = build_prompt(question, ctx)
    key = f"{LLM_MODEL}::{prompt}"
    if key in llm_cache:
        hit = llm_cache[key]
        return hit["text"], hit["seconds"]
    cfg = types.GenerateContentConfig(temperature=0.0, thinking_config=types.ThinkingConfig(thinking_budget=0))
    for attempt in range(retries):
        try:
            t0 = time.perf_counter()
            resp = llm.models.generate_content(model=LLM_MODEL, contents=prompt, config=cfg)
            seconds = time.perf_counter() - t0
            llm_cache[key] = {"text": resp.text.strip(), "seconds": seconds}
            json.dump(llm_cache, open(CACHE_PATH, "w"), indent=1, ensure_ascii=False)
            return resp.text.strip(), seconds
        except Exception as e:
            if "PerDay" in str(e):   # daily quota: retrying won't help
                raise RuntimeError(f"Daily Gemini quota for {LLM_MODEL} exhausted; re-run later (cached answers are kept)") from e
            wait = min(60, 5 * 2 ** attempt)
            print(f"  LLM error ({type(e).__name__}: {str(e)[:80]}), retrying in {wait}s")
            time.sleep(wait)
    raise RuntimeError("LLM call failed after retries")

def run_both(question):
    # Runs both pipelines end to end for one question.
    cands = baseline_retrieve(question)
    reranked = rerank(question, cands)
    top3_base, top3_rr = cands[:TOP_N], reranked[:TOP_N]
    assert build_prompt(question, top3_base).split("Rulebook excerpts:")[0] == build_prompt(question, top3_rr).split("Rulebook excerpts:")[0]
    ans1, t1 = generate(question, top3_base)
    ans2, t2 = generate(question, top3_rr)
    return dict(candidates=cands, reranked=reranked, answer1=ans1, answer2=ans2, llm_s1=t1, llm_s2=t2)

# show what the LLM actually receives (no API call here)
print(build_prompt("How many command tokens does each player start with?",
                   baseline_retrieve("How many command tokens does each player start with?")[:TOP_N])[:1500], "…")

You are a rules assistant for the board game Twilight Imperium: Fourth Edition.
Answer the question using ONLY the rulebook excerpts below. Do not use any outside knowledge of the game.
If the excerpts do not contain the information needed, reply exactly: "The provided excerpts do not cover this."
Answer in 2-4 sentences and cite the chunk ids you relied on, e.g. [Chunk 12].

Rulebook excerpts:
[Chunk 40, p.11] [Rule 19 Command Tokens] Command tokens are a currency that players use to perform actions and expand their fleets. 19.1 Each player begins the game with eight tokens on their command sheet: three in their tactic pool, three in their fleet pool, and two in their strategy pool. • Command tokens in the strategy and tactic pool are placed with the faction symbol faceup. • Command tokens in the fleet pool are placed with the ship silhouette faceup. 19.2 When a player gains a command token, they choose which of their three pools to place it in.

[Chunk 41, p.11] [Rule 19 Command Toke

## 6. Stress tests: where does the baseline fail?

Each candidate query is tagged with a failure category and **gold marker(s)**: verbatim phrases from the rulebook that contain the real
answer. The gold chunk ids are resolved by searching for these phrases, so the labels stay correct if the chunking changes.

Categories:
* `keyword_trap`: the query's words match a different rule strongly, while the true answer uses other terminology.
* `exception`: a general rule exists, but the answer is a specific restriction or “except when…” clause elsewhere.
* `negation`: “cannot / not allowed / prohibited” questions, where embeddings blur positive and negative statements.
* `buried`: any other case where the answer chunk sits at ranks 4–10.

**All** candidates are evaluated, including the ones where the baseline is already fine or where re-ranking makes things worse.

In [9]:
# Stress-test queries. `gold` = verbatim phrase(s) from the rulebook that contain the true answer;
# a chunk containing any of them counts as a correct retrieval.
CANDIDATES = [
    # ---------- keyword trap / distractor: query words match a different rule ----------
    dict(qid='KW01', cat='keyword_trap',
         q='How many ships am I allowed to keep in a single system?',
         gold=['maximum number of non-fighter ships that a player can have in a system']),
    dict(qid='KW02', cat='keyword_trap',
         q='When does the agenda phase start being played during the game?',
         gold=['the agenda phase is added to all subsequent game rounds']),
    dict(qid='KW03', cat='keyword_trap',
         q='After losing a space battle, where can my surviving ships run away to?',
         gold=['moves them to a single system that is adjacent to the active system']),
    dict(qid='KW04', cat='keyword_trap',
         q="What happens to a player's plastic pieces and cards when they are knocked out of the game?",
         gold=['faction sheet that matches that player’s faction or color are returned to the game box']),
    dict(qid='KW05', cat='keyword_trap',
         q='Can I use trade goods to pay for building new ships?',
         gold=['Spent resources must come from planets or trade goods']),
    dict(qid='KW06', cat='keyword_trap',
         q='Who gets a combat bonus when a space battle happens inside a nebula, the attacker or the defender?',
         gold=['the defender applies +1 to each combat roll of their ships during that combat']),
    dict(qid='KW07', cat='keyword_trap',
         q='What happens to my extra fighters if the carrier holding them gets destroyed?',
         gold=['that player must remove the excess units']),
    dict(qid='KW08', cat='keyword_trap',
         q='How does a player win the game?',
         gold=['The first player to gain 10 victory points wins the game']),
    dict(qid='KW09', cat='keyword_trap',
         q="Two players are tied on victory points when the last objective can't be revealed. Does the speaker decide the winner?",
         gold=['the tied player who is first in initiative order is the winner']),
    dict(qid='KW10', cat='keyword_trap',
         q='If two players both reach 10 points at the same moment, who wins?',
         gold=['the player who is earliest in initiative order among those players is the winner']),
    dict(qid='KW11', cat='keyword_trap',
         q='When I trade with someone, does the swap have to be fair?',
         gold=['A transaction does not have to be even']),
    dict(qid='KW12', cat='keyword_trap',
         q='Once we finished exchanging goods, can we take the trade back?',
         gold=['After the components are traded, the transaction cannot be undone']),
    dict(qid='KW13', cat='keyword_trap',
         q='Can I choose to just skip voting on an agenda?',
         gold=['A player may choose to abstain by not casting any votes']),
    dict(qid='KW14', cat='keyword_trap',
         q='Can I give my commodities to a neighbor for free without getting anything back?',
         gold=['give components without receiving something in return']),
    dict(qid='KW15', cat='keyword_trap',
         q='Can I swap action cards with my neighbor?',
         gold=['cannot exchange other types of cards or tokens']),
    dict(qid='KW16', cat='keyword_trap',
         q='Can other players look at my secret objectives?',
         gold=['hidden from all other players until it is scored']),
    dict(qid='KW17', cat='keyword_trap',
         q='How can I take the speaker token from the current speaker?',
         gold=['chooses any player other than the current speaker to gain the speaker token']),
    dict(qid='KW18', cat='keyword_trap',
         q='Is there a limit to how much I can trade with one neighbor on my turn?',
         gold=['they may resolve up to one transaction with each of their neighbors']),
    dict(qid='KW19', cat='keyword_trap',
         q='Where do my destroyed ships go?',
         gold=['it is removed from the game board and returned to their reinforcements']),
    # ---------- exceptions & corner cases: a specific clause overrides the general rule ----------
    dict(qid='EX01', cat='exception',
         q='Do I have to spend a strategy token to use the secondary ability of the Leadership card?',
         gold=['does not spend a command token to resolve the secondary ability of the “Leadership”']),
    dict(qid='EX02', cat='exception',
         q='Can I make a transaction with a player who is not my neighbor?',
         gold=['Players do not need to be neighbors to perform these transactions']),
    dict(qid='EX03', cat='exception',
         q='Can I still score public objectives if an enemy has captured one of my home system planets?',
         gold=['cannot score public objectives if that player does not control each planet in their home system']),
    dict(qid='EX04', cat='exception',
         q='Can a war sun bombard a planet protected by a PDS with planetary shield?',
         gold=['A war sun can use its “Bombardment” ability against planets that contain units']),
    dict(qid='EX05', cat='exception',
         q="If enemy ships are in my space dock's system, can I still produce infantry there?",
         gold=['can still use a blockaded unit to produce ground forces', 'Ground forces can still be produced']),
    dict(qid='EX06', cat='exception',
         q='Does a ship that ends its movement inside a gravity rift have to roll a die?',
         gold=['one die is rolled immediately before it exits the gravity rift system']),
    dict(qid='EX07', cat='exception',
         q='Do ground forces in the space area count against capacity while a space combat is happening?',
         gold=['do not count against capacity during combat']),
    dict(qid='EX08', cat='exception',
         q='If the speaker is eliminated, who receives the speaker token?',
         gold=['If the speaker becomes eliminated, the speaker token passes']),
    dict(qid='EX09', cat='exception',
         q='How many objectives can I score during a single combat in the action phase?',
         gold=['they can only score one objective during or after each combat']),
    dict(qid='EX10', cat='exception',
         q='I want to produce a unit but all of them are already on the board. What can I do?',
         gold=['that player can remove a unit from any system that does not contain one of their command tokens']),
    dict(qid='EX11', cat='exception',
         q="I'm the active player. Am I allowed to trade with my neighbor in the middle of a space combat?",
         gold=['A player can resolve a transaction at any time during their turn, even during a combat']),
    dict(qid='EX12', cat='exception',
         q='If I overpay resources for a unit, do I keep the leftover resources?',
         gold=['Any resources spent in excess of a unit’s cost are lost']),
    dict(qid='EX13', cat='exception',
         q='Do dice get rolled for the infantry being carried when my carrier leaves a gravity rift?',
         gold=['Dice are not rolled for units that are being transported']),
    dict(qid='EX14', cat='exception',
         q='I announced a retreat but my opponent lost all their ships before the retreat step. Do my ships still retreat?',
         gold=['the combat immediately ends and the retreat does not occur']),
    dict(qid='EX15', cat='exception',
         q='We started with six players and two got eliminated. Do we now each pick two strategy cards?',
         gold=['the players continue to select only one strategy card during the strategy phase']),
    dict(qid='EX16', cat='exception',
         q='A player has lost every planet but still has a space dock on the board. Are they out of the game?',
         gold=['A player is eliminated when they meet all of the following three conditions']),
    dict(qid='EX17', cat='exception',
         q="Can I make a deal with someone who isn't my neighbor?",
         gold=['Players can make deals with each other at any time, even if they are not neighbors']),
    dict(qid='EX18', cat='exception',
         q='Can my PDS fire at ships in a neighboring system instead of its own?',
         gold=['they can use the “Space Cannon” ability of their PDS units that are in systems that are adjacent to the active system']),
    dict(qid='EX19', cat='exception',
         q='Do action cards that give +1 to combat rolls also improve my bombardment rolls?',
         gold=['Game effects that reroll, modify, or otherwise affect combat rolls do not affect bombardment rolls']),
    dict(qid='EX20', cat='exception',
         q='Can my PDS still shoot during space cannon offense if no ships moved this turn?',
         gold=['This ability can be used even if no ships were moved']),
    dict(qid='EX21', cat='exception',
         q='My ship starts its turn sitting in a nebula. How far can it move?',
         gold=['treats its move value as “1” for the duration of that step']),
    dict(qid='EX22', cat='exception',
         q='If I gained a point from a law and that law is later discarded, do I lose the point?',
         gold=['that player does not lose that victory point']),
    dict(qid='EX23', cat='exception',
         q='We are playing a longer game on the other side of the track. How many points do we need to win?',
         gold=['the game ends and a player wins when they have 14 victory points instead of 10']),
    dict(qid='EX24', cat='exception',
         q='An enemy moved infantry onto my planet where I have no units left. Who controls it now?',
         gold=['A player loses control of a planet if they no longer have units on it and another player has units on it']),
    dict(qid='EX25', cat='exception',
         q="Do I have to use all of a planet's influence when I exhaust it to vote, or can I use part of it?",
         gold=['that player must cast the full amount of votes provided by that planet']),
    dict(qid='EX26', cat='exception',
         q='Does a player have to be the active player to fire their PDS?',
         gold=['A player is not required to be the active player to use their “Space Cannon” ability']),
    dict(qid='EX27', cat='exception',
         q="I meet the requirement on a public objective card. Is there any situation where I still can't score it?",
         gold=['cannot score public objectives if that player does not control each planet in their home system']),
    # ---------- negation & constraints: cannot / not allowed / prohibited ----------
    dict(qid='NG01', cat='negation',
         q='Which units are not allowed to be moved or carried by ships?',
         gold=['Structures cannot move or be transported']),
    dict(qid='NG02', cat='negation',
         q='Is it prohibited for ships to pass through a nebula on the way to another system?',
         gold=['A ship cannot move through a nebula']),
    dict(qid='NG03', cat='negation',
         q='Am I not allowed to land ground forces on Mecatol Rex at the start of the game?',
         gold=['players cannot commit ground forces to land on Mecatol Rex until the custodians token is removed']),
    dict(qid='NG04', cat='negation',
         q='If a game effect removes my unit from the board, is that unit considered destroyed?',
         gold=['it is not treated as being destroyed']),
    dict(qid='NG05', cat='negation',
         q='What items are players not allowed to exchange in a transaction?',
         gold=['cannot exchange other types of cards or tokens']),
    dict(qid='NG06', cat='negation',
         q='Can my carrier pick up fighters from a system that already has one of my command tokens?',
         gold=['cannot be picked up from a system that contains one of their faction’s command tokens']),
    dict(qid='NG07', cat='negation',
         q='Is a player prohibited from scoring a secret objective that another player revealed?',
         gold=['a player cannot score secret objectives revealed by other players']),
    dict(qid='NG08', cat='negation',
         q="Can my ships fly through a system that contains another player's ships?",
         gold=['cannot move through a system that contains ships that are controlled by another player']),
    dict(qid='NG09', cat='negation',
         q='Can I spend trade goods to get extra votes on an agenda?',
         gold=['Trade goods cannot be spent to cast votes']),
    dict(qid='NG10', cat='negation',
         q='Can I split my votes between two different outcomes of the same agenda?',
         gold=['cannot cast votes for multiple outcomes of the same agenda']),
    dict(qid='NG11', cat='negation',
         q='The defender announced a retreat this round. Am I as the attacker still allowed to announce one?',
         gold=['If the defender announces a retreat, the attacker cannot announce a retreat']),
    dict(qid='NG12', cat='negation',
         q='Can a binding deal include a promise to play an action card later?',
         gold=['They cannot be part of a binding deal']),
    dict(qid='NG13', cat='negation',
         q='Can my dreadnought use sustain damage to cancel a hit from anti-fighter barrage?',
         gold=['a player cannot use a dreadnought’s “Sustain Damage” ability to cancel a hit from “Anti-Fighter Barrage.”']),
    dict(qid='NG14', cat='negation',
         q='Can sustain damage save a ship from an effect that directly destroys it?',
         gold=['cannot be used to cancel an effect that directly destroys a unit']),
    dict(qid='NG15', cat='negation',
         q='Can I move a ship into a nebula that is not the system I activated?',
         gold=['A ship can only move into a nebula if it is the active system']),
    dict(qid='NG16', cat='negation',
         q='Can I gain control of a planet that I already own to trigger its abilities again?',
         gold=['A player cannot gain control of a planet that they already control']),
    dict(qid='NG17', cat='negation',
         q='If I exceed my capacity because the ship carrying units dies mid-battle, are the excess fighters removed immediately?',
         gold=['do not count against capacity during combat']),
    # ---------- other: answer is a small clause inside a chunk about something else ----------
    dict(qid='BR01', cat='buried',
         q='What happens to the ground forces being carried if their ship is lost in a gravity rift?',
         gold=['Units that are being transported are removed from the board if the ship transporting them', 'Units being transported by a ship that is removed from the board by a gravity rift']),
    dict(qid='BR02', cat='buried',
         q='How many PDS units can I place on a single planet?',
         gold=['maximum of two PDS units on each planet']),
    dict(qid='BR03', cat='buried',
         q='If I only want one fighter, do I still have to pay the full cost?',
         gold=['A player can choose to produce only one unit; however, they must still pay the entire cost']),
    dict(qid='BR04', cat='buried',
         q='Do units on planets count toward my fleet pool limit?',
         gold=['Units that are on planets or that count against a player’s capacity do not count against']),
    dict(qid='BR05', cat='buried',
         q='Is there a maximum number of secret objectives a player can hold?',
         gold=['up to three total scored and unscored secret objectives']),
    dict(qid='BR06', cat='buried',
         q='After my ships retreat to an adjacent system, do I have to put anything in that system?',
         gold=['must place a command token from their reinforcements in the system to which their units retreated']),
    dict(qid='BR07', cat='buried',
         q='The Ghosts of Creuss player was knocked out. Do their wormholes leave the board?',
         gold=['their wormhole tokens remain on the game board for the remainder of the game']),
    dict(qid='BR08', cat='buried',
         q='When I take over a planet, can I use its resources right away in the same turn?',
         gold=['place it in their play area; that card is exhausted']),
    dict(qid='BR09', cat='buried',
         q='If my opponent bombards with more hits than I have infantry on the planet, what happens to the extra hits?',
         gold=['the excess hits have no effect']),
    dict(qid='BR10', cat='buried',
         q="Does an ability using the word 'cannot' override other abilities that would allow it?",
         gold=['If an ability uses the word “cannot,” that effect is absolute']),
    dict(qid='BR11', cat='buried',
         q="If a 'when' ability and an 'after' ability trigger on the same event, which one is resolved first?",
         gold=['Effects that occur “when” an event happens take priority over effects that occur “after”']),
    dict(qid='BR12', cat='buried',
         q='If I leave a planet empty, do I still keep it?',
         gold=['A player can control a planet that they do not have any units on']),
    dict(qid='BR13', cat='buried',
         q='Can I vote for myself when an agenda asks to elect a player?',
         gold=['When electing a player, a player can cast votes for themselves']),
    dict(qid='BR14', cat='buried',
         q='If I have a damaged war sun, is it weaker in combat?',
         gold=['A damaged unit does not have reduced capabilities']),
]

In [10]:
def find_chunks(phrase):
    # Ids of chunks that contain a phrase (used to resolve gold markers / browse the corpus).
    return [c["id"] for c in chunks if phrase.lower() in c["text"].lower()]

def gold_ids(item):
    ids = sorted({i for m in item["gold"] for i in find_chunks(m)})
    assert ids, f"gold marker not found for {item['qid']}"
    return ids

# full-corpus bi-encoder rank of every chunk (to see how deep a gold chunk is, even beyond the top 10)
def full_bi_rank(query):
    order = np.argsort(-(chunk_emb @ bi_encoder.encode(query, normalize_embeddings=True)))
    return {int(cid): r + 1 for r, cid in enumerate(order)}

def outcome(b, r):
    if b <= TOP_N and r is not None and r <= TOP_N: return "hit in both"
    if b > TOP_N and r is not None and r <= TOP_N:  return "FIXED by re-ranking"
    if b <= TOP_N:                                  return "BROKEN by re-ranking"
    if b > TOP_K:                                   return "miss: gold not in top-10"
    return "miss in both"

rows = []
for item in CANDIDATES:
    g = gold_ids(item)
    cands = baseline_retrieve(item["q"])
    rr = rerank(item["q"], cands)
    fr = full_bi_rank(item["q"])
    best = min(g, key=lambda i: fr[i])                         # the gold chunk the bi-encoder ranks highest
    ce = {c["chunk_id"]: c["ce_rank"] for c in rr}
    ce_best = min((ce[i] for i in g if i in ce), default=None)
    rows.append(dict(qid=item["qid"], category=item["cat"], query=item["q"], gold_chunks=g,
                     bi_rank=fr[best], ce_rank=ce_best, outcome=outcome(fr[best], ce_best),
                     rank_shift=f"#{fr[best]} → #{ce_best}" if ce_best else f"#{fr[best]} → (not in top-10)",
                     base_top3=[c["chunk_id"] for c in cands[:TOP_N]], rerank_top3=[c["chunk_id"] for c in rr[:TOP_N]]))

shifts = pd.DataFrame(rows).astype({"ce_rank": "Int64"})
shifts.to_csv(RESULTS_DIR / "rank_shifts.csv", index=False)
print(shifts.outcome.value_counts().to_string(), "\n")
shifts[["qid", "category", "query", "gold_chunks", "rank_shift", "outcome"]]

outcome
hit in both                 56
FIXED by re-ranking         10
miss: gold not in top-10     6
miss in both                 5 



,qid,category,query,gold_chunks,rank_shift,outcome
0,KW01,keyword_trap,How many ships am I allowed to keep in a single system?,[69],#3 → #2,hit in both
1,KW02,keyword_trap,When does the agenda phase start being played during the game?,[50],#41 → (not in top-10),miss: gold not in top-10
2,KW03,keyword_trap,"After losing a space battle, where can my surviving ships run away to?",[128],#1 → #1,hit in both
3,KW04,keyword_trap,What happens to a player's plastic pieces and cards when they are knocked out of the game?,[57],#1 → #1,hit in both
4,KW05,keyword_trap,Can I use trade goods to pay for building new ships?,[110],#13 → (not in top-10),miss: gold not in top-10
...,...,...,...,...,...,...
72,BR10,buried,Does an ability using the word 'cannot' override other abilities that would allow it?,[1],#2 → #2,hit in both
73,BR11,buried,"If a 'when' ability and an 'after' ability trigger on the same event, which one is resolved first?",[3],#1 → #1,hit in both
74,BR12,buried,"If I leave a planet empty, do I still keep it?",[53],#3 → #3,hit in both
75,BR13,buried,Can I vote for myself when an agenda asks to elect a player?,[23],#1 → #1,hit in both


In [11]:
# outcome counts per category
pd.crosstab(shifts.category, shifts.outcome, margins=True)

outcome,FIXED by re-ranking,hit in both,miss in both,miss: gold not in top-10,All
category,,,,,
buried,2,11,1,0,14
exception,2,22,0,3,27
keyword_trap,4,9,4,2,19
negation,2,14,0,1,17
All,10,56,5,6,77


In [12]:
# cases where the cross-encoder moved the gold chunk DOWN (re-ranking is not always an improvement)
moved_down = shifts[shifts.ce_rank.notna() & (shifts.ce_rank > shifts.bi_rank)]
moved_down[["qid", "category", "query", "rank_shift", "outcome"]]

,qid,category,query,rank_shift,outcome
6,KW07,keyword_trap,What happens to my extra fighters if the carrier holding them gets destroyed?,#4 → #8,miss in both
12,KW13,keyword_trap,Can I choose to just skip voting on an agenda?,#4 → #5,miss in both
18,KW19,keyword_trap,Where do my destroyed ships go?,#5 → #7,miss in both
20,EX02,exception,Can I make a transaction with a player who is not my neighbor?,#1 → #3,hit in both
30,EX12,exception,"If I overpay resources for a unit, do I keep the leftover resources?",#2 → #3,hit in both
34,EX16,exception,A player has lost every planet but still has a space dock on the board. Are they out of the game?,#1 → #2,hit in both
54,NG09,negation,Can I spend trade goods to get extra votes on an agenda?,#1 → #2,hit in both
76,BR14,buried,"If I have a damaged war sun, is it weaker in combat?",#5 → #8,miss in both


### 6.1 Detailed rank movement for the failure cases

For each query that re-ranking **fixed**, the table shows all 10 candidates with their bi-encoder rank, cross-encoder rank and scores.
★ marks the gold chunk.

In [13]:
def rank_table(qid):
    item = next(x for x in CANDIDATES if x["qid"] == qid)
    g = gold_ids(item)
    rr = rerank(item["q"], baseline_retrieve(item["q"]))
    df = pd.DataFrame(rr)[["chunk_id", "bi_rank", "ce_rank", "bi_score", "ce_score"]]
    df["gold"] = df.chunk_id.map(lambda i: "★" if i in g else "")
    df["section"] = df.chunk_id.map(lambda i: chunks[i]["section"])
    df["text"] = df.chunk_id.map(lambda i: chunks[i]["text"][:110] + "…")
    return df.sort_values("bi_rank").round(3)

FAILURE_QIDS = shifts.loc[shifts.outcome == "FIXED by re-ranking", "qid"].tolist()
print("queries fixed by re-ranking:", FAILURE_QIDS)
for qid in FAILURE_QIDS:
    q = next(x["q"] for x in CANDIDATES if x["qid"] == qid)
    print(f"\n=== {qid}: {q}")
    display(rank_table(qid))

queries fixed by re-ranking: ['KW12', 'KW14', 'KW15', 'KW18', 'EX11', 'EX27', 'NG11', 'NG14', 'BR07', 'BR08']

=== KW12: Once we finished exchanging goods, can we take the trade back?


,chunk_id,bi_rank,ce_rank,bi_score,ce_score,gold,section,text
5,38,1,6,0.509,-6.760,,Rule 20 Commodities,[Rule 20 Commodities] 20.5 Players can trade commodities following the rules for transactions. When a player r…
4,46,2,5,0.507,-6.267,,Rule 20 Commodities,[Rule 20 Commodities] 20.6 Any game effect that instructs a player to give a commodity to another player cause…
1,161,3,2,0.485,-2.388,,Rule 82 Trade Goods,[Rule 82 Trade Goods] A trade good represents a player’s buying and trading power beyond their planet’s raw re…
8,203,4,9,0.481,-8.224,,Errata/FAQ General,[Errata/FAQ General] Q: When the player resolving the “Trade” strategy card selects another player to resolve …
2,162,5,3,0.474,-4.968,,Rule 82 Trade Goods,"[Rule 82 Trade Goods] 82.6 When a player receives a commodity token from another player, the player who receiv…"
0,170,6,1,0.443,-0.454,★,Rule 83 Transactions,"[Rule 83 Transactions] A transaction is a way for a player to exchange commodities, trade goods, and promissor…"
9,119,7,10,0.426,-8.483,,Rule 61 Readied,"[Rule 61 Readied] Cards have a readied state, which indicates that a player can exhaust or resolve the abiliti…"
6,165,8,7,0.400,-6.853,,Rule 81 Trade (Strategy Card),[Rule 81 Trade (Strategy Card)] The “Trade” strategy card allows players to gain trade goods and replenish com…
3,160,9,4,0.369,-6.144,,Rule 81 Trade (Strategy Card),[Rule 81 Trade (Strategy Card)] 81.4 STEP 3— The active player chooses any number of other players. Those play…
7,37,10,8,0.366,-7.899,,Rule 20 Commodities,[Rule 20 Commodities] Commodities represent goods that are plentiful for their own faction and are desired by …



=== KW14: Can I give my commodities to a neighbor for free without getting anything back?


,chunk_id,bi_rank,ce_rank,bi_score,ce_score,gold,section,text
4,46,1,5,0.479,-7.967,,Rule 20 Commodities,[Rule 20 Commodities] 20.6 Any game effect that instructs a player to give a commodity to another player cause…
5,38,2,6,0.453,-8.622,,Rule 20 Commodities,[Rule 20 Commodities] 20.5 Players can trade commodities following the rules for transactions. When a player r…
6,37,3,7,0.412,-9.343,,Rule 20 Commodities,[Rule 20 Commodities] Commodities represent goods that are plentiful for their own faction and are desired by …
7,162,4,8,0.359,-10.642,,Rule 82 Trade Goods,"[Rule 82 Trade Goods] 82.6 When a player receives a commodity token from another player, the player who receiv…"
2,203,5,3,0.332,-5.611,,Errata/FAQ General,[Errata/FAQ General] Q: When the player resolving the “Trade” strategy card selects another player to resolve …
0,170,6,1,0.319,-2.232,★,Rule 83 Transactions,"[Rule 83 Transactions] A transaction is a way for a player to exchange commodities, trade goods, and promissor…"
8,161,7,9,0.301,-10.857,,Rule 82 Trade Goods,[Rule 82 Trade Goods] A trade good represents a player’s buying and trading power beyond their planet’s raw re…
1,171,8,2,0.294,-5.032,,Rule 83 Transactions,[Rule 83 Transactions] 83.5 Players can resolve a transaction as part of a deal. 83.6 While resolving each age…
3,119,9,4,0.286,-7.569,,Rule 61 Readied,"[Rule 61 Readied] Cards have a readied state, which indicates that a player can exhaust or resolve the abiliti…"
9,115,10,10,0.258,-11.378,,Rule 64 Resources,"[Rule 64 Resources] Resources represent a planet’s material value and industry. Many game effects, such as pro…"



=== KW15: Can I swap action cards with my neighbor?


...

,chunk_id,bi_rank,ce_rank,bi_score,ce_score,gold,section,text
1,192,1,2,0.565,2.587,,Errata/FAQ Direct Hit,[Errata/FAQ Direct Hit] The text of the “Direct Hit” action card should read as follows: “After another player…
5,152,2,6,0.490,-2.454,,Rule 76 Sustain Damage (Unit Ability),[Rule 76 Sustain Damage (Unit Ability)] Some units have the “Sustain Damage” ability. Immediately before a pla…
2,60,3,3,0.475,-1.376,,Rule 29 Destroyed,"[Rule 29 Destroyed] Various game effects can cause a unit to be destroyed. When a player’s unit is destroyed, …"
0,153,4,1,0.446,2.800,★,Rule 76 Sustain Damage (Unit Ability),[Rule 76 Sustain Damage (Unit Ability)] 76.4 A unit can use its “Sustain Damage” ability any time a hit is pro…
3,127,5,4,0.329,-2.031,,Rule 67 Space Combat,"[Rule 67 Space Combat] After resolving the “Space Cannon Offense” step of a tactical action, if two players ha…"
6,213,6,7,0.325,-5.442,,Errata/FAQ Action Cards,[Errata/FAQ Action Cards] Q: Do infantry involved in a space combat due to the effects of “The Alastor” flagsh…
4,209,7,5,0.321,-2.296,,Errata/FAQ Action Cards,[Errata/FAQ Action Cards] Q: Can the “Shields Holding” action card be used to cancel hits from abilities that …
9,96,8,10,0.295,-8.069,,Rule 52 Objective Cards,"[Rule 52 Objective Cards] 52.9 If an objective requires a player to destroy one or more units, those units can…"
8,208,9,9,0.292,-6.481,,Errata/FAQ Action Cards,"[Errata/FAQ Action Cards] Q: Is the “Skilled Retreat” action card considered to be a retreat, and can it be ca…"
7,123,10,8,0.283,-6.474,,Rule 66 Space Cannon (Unit Ability),[Rule 66 Space Cannon (Unit Ability)] that are adjacent to the active system. The hits are still assigned to u…



=== BR07: The Ghosts of Creuss player was knocked out. Do their wormholes leave the board?


,chunk_id,bi_rank,ce_rank,bi_score,ce_score,gold,section,text
1,222,1,2,0.571,2.548,,Errata/FAQ The Ghosts Of Creuss,[Errata/FAQ The Ghosts Of Creuss] Q: Can the Creuss Promissory note still be used if the Creuss have not resea…
3,174,2,4,0.486,0.351,,Rule 89 Wormholes,[Rule 89 Wormholes] Some systems contain wormholes. Systems that contain identical wormholes are adjacent. 89.…
2,211,3,3,0.409,0.572,,Errata/FAQ Action Cards,"[Errata/FAQ Action Cards] Q: During a revote initiated by the “Miscount Disclosed” agenda, does the law that i…"
0,58,4,1,0.391,3.640,★,Rule 31 Elimination,[Rule 31 Elimination] 31.9 If a game that started with five or more players becomes a game with four or fewer …
5,185,5,6,0.386,-7.147,,Game Setup,"[Game Setup] starting with the speaker and proceeding clockwise, each player places one system tile faceup in …"
6,63,6,7,0.369,-8.130,,Rule 31 Elimination,[Rule 31 Elimination] A player who is eliminated is no longer part of the game. 31.1 A player is eliminated wh…
4,64,7,5,0.341,-2.146,,Rule 35 Game Board,[Rule 35 Game Board] The game board consists of all system tiles in play. 35.1 The game board consists of all …
9,66,8,10,0.320,-10.205,,Rule 37 Gravity Rift,[Rule 37 Gravity Rift] A gravity rift is an anomaly that affects movement. 37.1 A ship that will move out of o…
7,86,9,8,0.309,-10.133,,Rule 46 Mecatol Rex,[Rule 46 Mecatol Rex] Mecatol Rex is the planet placed in the center of the game board during setup. 46.1 Duri…
8,50,10,9,0.307,-10.196,,Rule 26 Custodians Token,[Rule 26 Custodians Token] 26.1 Units can move into the system that contains Mecatol Rex following normal rule…



=== BR08: When I take over a planet, can I use its resources right away in the same turn?


,chunk_id,bi_rank,ce_rank,bi_score,ce_score,gold,section,text
0,115,1,1,0.547,-6.084,,Rule 64 Resources,"[Rule 64 Resources] Resources represent a planet’s material value and industry. Many game effects, such as pro…"
4,97,2,5,0.455,-8.464,,Rule 55 Planets,"[Rule 55 Planets] 55.3 A planet’s trait has no inherent effects, but some game effects refer to a planet’s tra…"
3,67,3,4,0.409,-8.281,,Rule 32 Exhausted,[Rule 32 Exhausted] 32.3 A player exhausts their planet cards to spend either the resources or influence on th…
7,54,4,8,0.405,-9.993,,Rule 24 Control,[Rule 24 Control] 24.5 A player loses control of a planet if they no longer have units on it and another playe…
2,103,5,3,0.402,-7.889,,Rule 55 Planets,[Rule 55 Planets] Planets provide players with resources and influence. Planets are on system tiles and each h…
1,53,6,2,0.389,-7.003,★,Rule 24 Control,"[Rule 24 Control] 24.1 When a player gains control of a planet, they take the planet card that corresponds to …"
9,191,7,10,0.371,-11.032,,Errata/FAQ Diplomacy,[Errata/FAQ Diplomacy] The text of the “Diplomacy” strategy card’s primary ability should read as follows: “Ch…
5,112,8,6,0.356,-8.921,,Rule 61 Readied,[Rule 61 Readied] 61.1 A card that is readied is placed faceup in a player’s play area; a card that is exhaust…
6,77,9,7,0.350,-9.749,,Rule 44 Invasion,[Rule 44 Invasion] 44.5 STEP 5—ESTABLISH CONTROL: The active player gains control of each planet they committe…
8,12,10,9,0.340,-10.831,,Rule 2 Action Cards,"[Rule 2 Action Cards] 2.8 If an action card is canceled, that card has no effect and is discarded. RELATED TOP…"


### 6.2 Answer 1 vs Answer 2

Both pipelines are run end to end (same prompt, same LLM) on every query that re-ranking fixed, and on every query it broke, if any.
`gold_in_ctx` shows whether the context actually contained the rule. Extra counter-examples can be added to `COUNTER_EXAMPLES`
(e.g. `"NG09"`), but each query costs 2 Gemini calls, and the free tier allows about 20 per model per day.

In [14]:
COUNTER_EXAMPLES = []   # optional extra query ids to run through the LLM
SHOWCASE = FAILURE_QIDS + shifts.loc[shifts.outcome == "BROKEN by re-ranking", "qid"].tolist() + COUNTER_EXAMPLES
answers = []
for qid in SHOWCASE:
    item = next(x for x in CANDIDATES if x["qid"] == qid)
    g = gold_ids(item)
    out = run_both(item["q"])
    answers.append(dict(qid=qid, category=item["cat"], query=item["q"], gold_chunks=g,
                        base_top3=[c["chunk_id"] for c in out["candidates"][:TOP_N]],
                        rerank_top3=[c["chunk_id"] for c in out["reranked"][:TOP_N]],
                        gold_in_ctx_1=any(c["chunk_id"] in g for c in out["candidates"][:TOP_N]),
                        gold_in_ctx_2=any(c["chunk_id"] in g for c in out["reranked"][:TOP_N]),
                        answer1=out["answer1"], answer2=out["answer2"],
                        llm_s1=out["llm_s1"], llm_s2=out["llm_s2"]))
    time.sleep(4)   # stay under free-tier requests-per-minute limits

json.dump(answers, open(RESULTS_DIR / "answers.json", "w"), indent=1, ensure_ascii=False)
for a in answers:
    print(f"\n{'=' * 110}\n[{a['qid']} | {a['category']}] {a['query']}\n  gold chunks: {a['gold_chunks']}")
    print(f"\n  ANSWER 1 (baseline top-3 {a['base_top3']}, gold in context: {a['gold_in_ctx_1']}):\n  {a['answer1']}")
    print(f"\n  ANSWER 2 (re-ranked top-3 {a['rerank_top3']}, gold in context: {a['gold_in_ctx_2']}):\n  {a['answer2']}")


[KW12 | keyword_trap] Once we finished exchanging goods, can we take the trade back?
  gold chunks: [170]

  ANSWER 1 (baseline top-3 [38, 46, 161], gold in context: False):
  The provided excerpts do not cover this.

  ANSWER 2 (re-ranked top-3 [170, 161, 162], gold in context: True):
  No, the transaction cannot be undone. Once players agree on the terms and exchange the components, the trade cannot be taken back [Chunk 170].

[KW14 | keyword_trap] Can I give my commodities to a neighbor for free without getting anything back?
  gold chunks: [170]

  ANSWER 1 (baseline top-3 [46, 38, 37], gold in context: False):
  Players can trade commodities with other players by following the rules for transactions [Chunk 38]. When a player receives a commodity from another player, it is converted into a trade good on their command sheet [Chunk 38]. However, the provided excerpts do not contain the specific rules for transactions to state whether a trade requires receiving something in return.



In [15]:
# the gold rule text for each showcase query, for checking the answers against the source
for a in answers:
    print(f"\n[{a['qid']}]")
    for i in a["gold_chunks"]:
        print(f"  Chunk {i} (p.{chunks[i]['page']}): {chunks[i]['text']}")


[KW12]
  Chunk 170 (p.30): [Rule 83 Transactions] A transaction is a way for a player to exchange commodities, trade goods, and promissory notes. 83.1 During the active player’s turn, they may resolve up to one transaction with each of their neighbors. • A player can resolve a transaction at any time during their turn, even during a combat. 83.2 To resolve a transaction, a player gives any number of trade goods and commodities and up to one promissory note to a neighbor in exchange for any number of trade goods and commodities and up to one promissory note. 83.3 Players can exchange commodities, trade goods and promissory notes, but cannot exchange other types of cards or tokens. • The Emirates of Hacan can also exchange action cards with other players as part of their transactions. 83.4 A transaction does not have to be even. A player may exchange components of unequal value or give components without receiving something in return. • The players agree on the terms of the transaction 

## 7. Latency profiling

Each stage is timed separately with `time.perf_counter()`, after warm-up calls, over every candidate query × 20 repetitions:

* **Retrieval:** query embedding plus the Chroma top-10 search (both pipelines pay this).
* **Re-rank:** cross-encoder scoring of 10 pairs and the sort (Pipeline 2 only).
* **LLM generation:** taken from the real Gemini calls in §6.2. It is network-bound and the same for both pipelines.

In [16]:
def sync():
    if DEVICE == "cuda": torch.cuda.synchronize()

def time_stage(fn, reps=20):
    ts = []
    for _ in range(reps):
        sync(); t0 = time.perf_counter(); fn(); sync()
        ts.append((time.perf_counter() - t0) * 1000)
    return ts

queries = [x["q"] for x in CANDIDATES]
for q in queries[:3]:                      # warm-up (model graph / caches)
    rerank(q, baseline_retrieve(q))

ret_ms, rr_ms = [], []
for q in queries:
    cands = baseline_retrieve(q)
    ret_ms += time_stage(lambda: baseline_retrieve(q))
    rr_ms  += time_stage(lambda: rerank(q, cands))

llm_ms = [a["llm_s1"] * 1000 for a in answers] + [a["llm_s2"] * 1000 for a in answers]

def summarise(name, xs):
    return dict(stage=name, n=len(xs), mean_ms=np.mean(xs), std_ms=np.std(xs),
                p50_ms=np.percentile(xs, 50), p95_ms=np.percentile(xs, 95))

lat = pd.DataFrame([summarise("retrieval: embed query + top-10 search", ret_ms),
                    summarise("re-rank: cross-encoder on 10 pairs", rr_ms),
                    summarise("LLM generation (Gemini API)", llm_ms)]).round(2)
m = lat.set_index("stage").mean_ms
totals = pd.DataFrame([
    dict(pipeline="Pipeline 1 (baseline)", retrieval_ms=m.iloc[0], rerank_ms=0.0, llm_ms=m.iloc[2], total_ms=m.iloc[0] + m.iloc[2]),
    dict(pipeline="Pipeline 2 (re-rank)", retrieval_ms=m.iloc[0], rerank_ms=m.iloc[1], llm_ms=m.iloc[2], total_ms=m.iloc[0] + m.iloc[1] + m.iloc[2]),
]).round(2)
totals["retrieval_only_ms"] = (totals.retrieval_ms + totals.rerank_ms).round(2)
lat.to_csv(RESULTS_DIR / "latency.csv", index=False)
totals.to_csv(RESULTS_DIR / "latency_totals.csv", index=False)
print(f"device: {DEVICE} | torch threads: {torch.get_num_threads()}")
display(lat); display(totals)

device: cpu | torch threads: 6


,stage,n,mean_ms,std_ms,p50_ms,p95_ms
0,retrieval: embed query + top-10 search,1540,8.60,1.32,8.24,10.74
1,re-rank: cross-encoder on 10 pairs,1540,207.33,23.49,201.77,248.10
2,LLM generation (Gemini API),20,2597.79,1275.10,2132.09,5639.94


,pipeline,retrieval_ms,rerank_ms,llm_ms,total_ms,retrieval_only_ms
0,Pipeline 1 (baseline),8.6,0.00,2597.79,2606.39,8.60
1,Pipeline 2 (re-rank),8.6,207.33,2597.79,2813.72,215.93


## 8. Scaling: why not run the cross-encoder over everything?

This compares re-scoring the 10 candidates with scoring **every** chunk in the corpus with the cross-encoder, which is what “skip the
bi-encoder” would mean. A linear extrapolation to larger collections follows, because a cross-encoder must run one full transformer pass
per (query, document) pair at query time, and none of that work can be precomputed.

In [17]:
q = CANDIDATES[0]["q"]
all_pairs = [(q, t) for t in texts]
q_cands = baseline_retrieve(q)
ce_all_ms = time_stage(lambda: cross_encoder.predict(all_pairs, batch_size=64, show_progress_bar=False), reps=5)
ce_10_ms = time_stage(lambda: rerank(q, q_cands), reps=20)
bi_ms = time_stage(lambda: baseline_retrieve(q), reps=20)

per_pair_ms = np.mean(ce_all_ms) / len(texts)
scale = pd.DataFrame([
    dict(setting=f"bi-encoder top-10 search over {len(texts)} chunks", ms=np.mean(bi_ms)),
    dict(setting="cross-encoder on top-10 only", ms=np.mean(ce_10_ms)),
    dict(setting=f"cross-encoder on ALL {len(texts)} chunks", ms=np.mean(ce_all_ms)),
] + [dict(setting=f"cross-encoder on {n:,} chunks (extrapolated)", ms=per_pair_ms * n) for n in (10_000, 1_000_000)])
scale["seconds"] = (scale.ms / 1000).round(3)
scale.to_csv(RESULTS_DIR / "scaling.csv", index=False)
print(f"cross-encoder cost ≈ {per_pair_ms:.3f} ms per (query, chunk) pair on {DEVICE}")
scale.round(2)

cross-encoder cost ≈ 19.657 ms per (query, chunk) pair on cpu


,setting,ms,seconds
0,bi-encoder top-10 search over 225 chunks,7.99,0.01
1,cross-encoder on top-10 only,192.51,0.19
2,cross-encoder on ALL 225 chunks,4422.75,4.42
3,"cross-encoder on 10,000 chunks (extrapolated)",196566.77,196.57
4,"cross-encoder on 1,000,000 chunks (extrapolated)",19656677.37,19656.68


## 9. Summary tables (inputs to the report)

In [18]:
print("Rank shifts for the queries re-ranking fixed:")
display(shifts.loc[shifts.qid.isin(FAILURE_QIDS), ["qid", "category", "query", "gold_chunks", "rank_shift"]])
print("Latency (mean ms per query):")
display(totals)
print("Scaling:")
display(scale.round(2))

Rank shifts for the queries re-ranking fixed:


,qid,category,query,gold_chunks,rank_shift
11,KW12,keyword_trap,"Once we finished exchanging goods, can we take the trade back?",[170],#6 → #1
13,KW14,keyword_trap,Can I give my commodities to a neighbor for free without getting anything back?,[170],#6 → #1
14,KW15,keyword_trap,Can I swap action cards with my neighbor?,[170],#7 → #1
17,KW18,keyword_trap,Is there a limit to how much I can trade with one neighbor on my turn?,[170],#6 → #1
29,EX11,exception,I'm the active player. Am I allowed to trade with my neighbor in the middle of a space combat?,[170],#9 → #1
45,EX27,exception,I meet the requirement on a public objective card. Is there any situation where I still can't score it?,[91],#5 → #3
56,NG11,negation,The defender announced a retreat this round. Am I as the attacker still allowed to announce one?,[132],#7 → #1
59,NG14,negation,Can sustain damage save a ship from an effect that directly destroys it?,[153],#4 → #1
69,BR07,buried,The Ghosts of Creuss player was knocked out. Do their wormholes leave the board?,[58],#4 → #1
70,BR08,buried,"When I take over a planet, can I use its resources right away in the same turn?",[53],#6 → #2


Latency (mean ms per query):


,pipeline,retrieval_ms,rerank_ms,llm_ms,total_ms,retrieval_only_ms
0,Pipeline 1 (baseline),8.6,0.00,2597.79,2606.39,8.60
1,Pipeline 2 (re-rank),8.6,207.33,2597.79,2813.72,215.93


Scaling:


,setting,ms,seconds
0,bi-encoder top-10 search over 225 chunks,7.99,0.01
1,cross-encoder on top-10 only,192.51,0.19
2,cross-encoder on ALL 225 chunks,4422.75,4.42
3,"cross-encoder on 10,000 chunks (extrapolated)",196566.77,196.57
4,"cross-encoder on 1,000,000 chunks (extrapolated)",19656677.37,19656.68
